In [1]:
import os
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from dotenv import load_dotenv

/home/aryansharma/Desktop/Aryan Pendrive Data/Support-Integrity-Auditor/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

# Small encoder — full fine-tune. ~66M params, fits comfortably in 4GB, no quantization/LoRA needed.
# DistilBERT is a strong, well-supported default for short-text binary classification.
model_name = "distilbert-base-uncased"

print("="*50)
print(f"Loading {model_name} ...")
print("="*50)

tokenizer = AutoTokenizer.from_pretrained(model_name)

num_labels = 2
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
)
model = model.to("cuda")

print(f"Model loaded on: {model.device}")
print(f"Model dtype: {model.dtype}")

Loading distilbert-base-uncased ...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8318.40it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on: cuda:0
Model dtype: torch.float32


In [10]:
# No LoRA / no quantization — DistilBERT is small enough to fully fine-tune on 4GB.
n_total = sum(p.numel() for p in model.parameters())
print(f"Total params: {n_total/1e6:.1f}M (all trainable)")

Total params: 67.0M (all trainable)


In [27]:
training_args = TrainingArguments(
    output_dir="./distilbert_mismatch",
    num_train_epochs=50,                  # encoder fine-tune converges fast; watch eval F1
    per_device_train_batch_size=32,      # DistilBERT is tiny; 4GB handles this easily
    per_device_eval_batch_size=64,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=20,
    save_strategy="epoch",
    eval_strategy="epoch",               # must match save_strategy for load_best_model_at_end
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    learning_rate=2e-5,                  # standard full fine-tune LR (NOT 2e-4 — that was for LoRA)
    fp16=True,
    report_to="none",
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [28]:
def to_text(row):
    tic_desc = row["Ticket_Description"]
    tic_sub = row["Ticket_Subject"]
    categ = row["Issue_Category"]
    res = row["Resolution_Time_Hours"]
    sc = row["Satisfaction_Score"]

    txt = tic_sub + ": "+ tic_desc + " | " + "Issue Category: " + categ + " | " + "Resolution Time Taken In Hour: " + str(res) + " | "  + "Satisfactory Score: " + str(sc)

    return txt 

In [29]:
df = pd.read_csv("../dataset/preprocessed_data.csv")

In [30]:
df["TXT"] = df.apply(to_text,axis=1)

In [31]:
dataset = df[["TXT", "Is_Mismatch"]]

In [32]:
dataset

,TXT,Is_Mismatch
0,"Hours of operation - Individual: Hi Support, W...",1
1,"Data not syncing - Card: Hi Support, The appli...",1
2,"2FA issues - Question: Hi Support, How do I up...",0
3,"Login failed - Let: Hi Support, The dashboard ...",1
4,"Refund status - Attention: Hi Support, I have ...",1
...,...,...
19995,"Installation issue - Think: Hi Support, The ap...",1
19996,"Alert notification - Reality: Hi Support, I re...",0
19997,"Subscription upgrade - Spring: Hi Support, My ...",1
19998,"Suspicious charge - Even: Hi Support, I have b...",0


In [33]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

In [34]:
train_df, eval_df = train_test_split(dataset, test_size=0.2, random_state=42, stratify=dataset["Is_Mismatch"])

In [35]:
train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)

In [36]:
import numpy as np
from sklearn.metrics import accuracy_score,precision_recall_fscore_support

In [37]:
# Define metrics for evaluation
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    # Calculate metrics
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }


In [38]:
def tokenize_function(row):
    return tokenizer(
        row['TXT'],
        truncation=True,
        max_length=128   # tickets are ~55 tokens; 128 is plenty of headroom
    )

In [39]:
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 4000/4000 [00:00<00:00, 14897.81 examples/s]


In [40]:
tokenized_train = tokenized_train.remove_columns(['TXT'])
tokenized_eval = tokenized_eval.remove_columns(['TXT'])

In [41]:
tokenized_train = tokenized_train.rename_column("Is_Mismatch", 'labels')
tokenized_eval = tokenized_eval.rename_column("Is_Mismatch", 'labels')

# Set format for PyTorch
tokenized_train.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
tokenized_eval.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

# Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [42]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    # tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.556479,0.588470,0.666750,0.666201,0.665682,0.666750
2,0.592532,0.580915,0.669500,0.664931,0.662058,0.669500
3,0.583024,0.574903,0.677000,0.637570,0.653724,0.677000
4,0.559958,0.587900,0.665000,0.605070,0.634451,0.665000
5,0.565453,0.580622,0.669500,0.661145,0.657621,0.669500
6,0.543600,0.590848,0.672500,0.670157,0.668326,0.672500
7,0.551601,0.594390,0.659000,0.651159,0.647332,0.659000
8,0.511416,0.635384,0.639250,0.635537,0.632755,0.639250
9,0.458548,0.711229,0.647750,0.641890,0.638268,0.647750
10,0.386378,0.814657,0.633750,0.626451,0.622109,0.633750


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.46it/s]


KeyboardInterrupt: 

In [23]:
import torch
import gc

# 1. Basic cache clear - releases unused cached memory
torch.cuda.empty_cache()

# 2. Force Python garbage collection
gc.collect()

# 3. Synchronize all CUDA streams
torch.cuda.synchronize()

In [24]:
import torch

# Check current GPU memory state
def check_gpu_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3  # GB
        reserved = torch.cuda.memory_reserved() / 1024**3    # GB
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        
        # Free memory calculation
        free = total - allocated  # Not entirely accurate due to caching
        actually_free = total - reserved  # More accurate for "can I allocate more?"
        
        print(f"Total GPU Memory:     {total:.2f} GB")
        print(f"Allocated by PyTorch: {allocated:.2f} GB")
        print(f"Reserved (cached):    {reserved:.2f} GB")
        print(f"≈ Actually Free:      {actually_free:.2f} GB")
        print(f"PyTorch-reported free:{free:.2f} GB")
    else:
        print("CUDA not available")

check_gpu_memory()

Total GPU Memory:     3.68 GB
Allocated by PyTorch: 3.48 GB
Reserved (cached):    3.52 GB
≈ Actually Free:      0.16 GB
PyTorch-reported free:0.21 GB
